In [1]:
import pandas as pd
from collections import Counter 
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go



In [2]:
df_clinique = pd.read_csv("data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv",sep=";", dtype={"codepost":str})
saintCloud = [48.844864, 2.218416]
paris = [48.84363,2.344745 ]

In [3]:
df_clinique.columns

Index(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'pseudo_provisoire',
       'adresse', 'codepost', 'nom_commune_postal', 'requete', 'x', 'y',
       'score', 'trust_score', 'street', 'city', 'pc_city', 'ic_city',
       'code_dept', 'dept', 'reg', 'address', 'address_has_num_init',
       'address_has_num_geoloc', 'same_city', 'hostel', 'hosted',
       'date_geoloc', 'geometry', 'CODE_IRIS', 'INSEE_REG', 'CODE_DEPT',
       'patient_sexe', 'date_naissance', 'centre', 'ageaudiag', 'cancernum',
       'date_diag', 'topo_initiale_cim10', 'topo_initialelib', 'patho'],
      dtype='object')

# Calcul de distance

In [4]:
## Distance entre deux point d'une sphère => coordonnées exprimées en WGS84 (degrès décimaux): comprend un modèle de la terre  d'où le calcul de distance angulaire 
def calculer_distance_haversine(lat1, lon1, lat2, lon2):

    try : 
        # Rayon de la Terre en kilomètres
        R = 6371.0

        # Conversion des degrés en radians
        dLat = math.radians(lat2 - lat1)
        dLon = math.radians(lon2 - lon1)
        rLat1 = math.radians(lat1)
        rLat2 = math.radians(lat2)

        # Formule de Haversine
        a = math.sin(dLat / 2)**2 + math.cos(rLat1) * math.cos(rLat2) * math.sin(dLon / 2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        # Distance totale
        distance = R * c
        return distance
    except Exception as e : 
        print(f"Erreur lors du calcul de la distance : {e}")
        return np.nan
    

In [5]:
df_clinique.centre.unique()

array(['paris', 'saint-cloud'], dtype=object)

### DESCRIPTION DES EVENEMENTS SUR LE TERRITOIRE FRANCAIS

In [6]:
#importation des evenements médicaux sur le territoire Francais
df_em = pd.read_csv("../geocodeur/from_hegp/data/data_octobre_2023/medical_event/medical_events_all.csv", sep = ",")
df_em.groupby('EM').size().reset_index()


,EM,0
0,ap,1022
1,ep_mtev,694
2,ins,31
3,ira,420
4,pneumo,607
5,unap_appel,2090
6,unap_consult,1706


In [7]:
#evenements médicaux par patients uniques sur le territoire Francais
df_em.groupby('EM')['pseudo_provisoire'].nunique().reset_index()

,EM,pseudo_provisoire
0,ap,654
1,ep_mtev,493
2,ins,25
3,ira,351
4,pneumo,467
5,unap_appel,1659
6,unap_consult,1225


In [8]:
#liste de tous les patients uniques ayant fait des evenements medicaux
id_em =  df_em['pseudo_provisoire'].unique()
len(id_em)

3521

In [9]:
#liste des patients uniques de notre cohorte ayant fait des evenements medicaux 
patients_em = df_clinique[ df_clinique['pseudo_provisoire'].isin(id_em)].drop_duplicates('pseudo_provisoire', keep='first')
len(patients_em)

3284

In [10]:
#repartition par evenements medicaux des patients de notre cohorte 
patients_em = df_em.merge(patients_em)
patients_em.groupby('EM').size().reset_index()




,EM,0
0,ap,686
1,ep_mtev,652
2,ins,30
3,ira,403
4,pneumo,574
5,unap_appel,2068
6,unap_consult,1675


### DESCRIPTION DES EVENEMENTS EN IDF

In [11]:
# nombre d'evenements medicaux des patients en IDF
IDF_em = patients_em[patients_em['INSEE_REG']==11]
IDF_em.columns

Index(['EM', 'pseudo_provisoire', 'Date_debut', 'Unnamed: 0.2', 'Unnamed: 0.1',
       'Unnamed: 0', 'adresse', 'codepost', 'nom_commune_postal', 'requete',
       'x', 'y', 'score', 'trust_score', 'street', 'city', 'pc_city',
       'ic_city', 'code_dept', 'dept', 'reg', 'address',
       'address_has_num_init', 'address_has_num_geoloc', 'same_city', 'hostel',
       'hosted', 'date_geoloc', 'geometry', 'CODE_IRIS', 'INSEE_REG',
       'CODE_DEPT', 'patient_sexe', 'date_naissance', 'centre', 'ageaudiag',
       'cancernum', 'date_diag', 'topo_initiale_cim10', 'topo_initialelib',
       'patho'],
      dtype='object')

In [12]:
#repartition des evenements medicaux des patients en IDF
IDF_em.groupby('EM').size().reset_index()

,EM,0
0,ap,624
1,ep_mtev,586
2,ins,25
3,ira,369
4,pneumo,508
5,unap_appel,1821
6,unap_consult,1549


In [13]:
#Repartition des evenements médicaux par patients uniques en IDF
IDF_em.groupby('EM')['pseudo_provisoire'].nunique().reset_index()

,EM,pseudo_provisoire
0,ap,439
1,ep_mtev,418
2,ins,20
3,ira,304
4,pneumo,386
5,unap_appel,1446
6,unap_consult,1100


In [14]:
patients_em.to_csv("../geocodeur/from_hegp/data/data_octobre_2023/medical_event/patients_fr_all_EM.csv",sep=';')

In [15]:
# Séparation de la cohorte  en deux DataFrames
df_paris = patients_em[patients_em['centre'] == 'paris'].copy()
df_saintCloud = patients_em[patients_em['centre'] == 'saint-cloud'].copy()

# Ajout de colonnes 'distance_km' et 'distance_int' pour Paris
df_paris['distance_km'] = df_paris.apply(lambda row: calculer_distance_haversine(row['y'], row['x'], paris[0], paris[1]), axis=1)
df_paris['distance_int'] = df_paris['distance_km'].apply(
    lambda dist: '< 1km' if dist <= 1 else 
                 '1km < d <= 10km' if 1 < dist <= 10 else 
                 '10km < d <= 30km' if 10 < dist <= 30 else 
                 '30km < d <= 50km' if 30 < dist <= 50 else 
                 '50km < d <= 100km' if 50 < dist <= 100 else '> 100km'
)

# Ajout de colonnes 'distance_km' et 'distance_int' pour Saint-Cloud
df_saintCloud['distance_km'] = df_saintCloud.apply(lambda row: calculer_distance_haversine(row['y'], row['x'], saintCloud[0], saintCloud[1]), axis=1)
df_saintCloud['distance_int'] = df_saintCloud['distance_km'].apply(
    lambda dist: '< 1km' if dist <= 1 else 
                 '1km < d <= 10km' if 1 < dist <= 10 else 
                 '10km < d <= 30km' if 10 < dist <= 30 else 
                 '30km < d <= 50km' if 30 < dist <= 50 else 
                 '50km < d <= 100km' if 50 < dist <= 100 else '> 100km'
)

# Modification de la pathologie pour un indice spécifique (si applicable)
if 24201 in df_clinique.index:
    df_clinique.at[24201, 'patho'] = "Autre"




In [16]:
df_paris.to_csv('../geocodeur/from_hegp/data/data_octobre_2023/medical_event/patients_EM_dist_centre_Paris.csv',sep=";")
df_saintCloud.to_csv('../geocodeur/from_hegp/data/data_octobre_2023/medical_event/patients_EM_dist_centre_stCloud.csv', sep=";")

In [17]:
df_clinique[df_clinique['pseudo_provisoire']==24201]


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,...,CODE_DEPT,patient_sexe,date_naissance,centre,ageaudiag,cancernum,date_diag,topo_initiale_cim10,topo_initialelib,patho
22157,23479,24200,24200,24201,4 RUE DES SOURCES FORGES ...,77630.0,SAINT MARTIN EN BIERE,4 RUE DES SOURCES FORGES ...,2.560988,48.440581,...,77,F,1968-11,paris,49.0,1.0,2018-08-15,C86,NaN,Hemato


In [18]:

df_dist_paris = pd.DataFrame(df_paris.drop_duplicates(subset='pseudo_provisoire', keep='first').groupby('distance_int').size(), columns = ['count'])
df_dist_paris = df_dist_paris.reindex(['< 1km', '1km < d <= 10km','10km < d <= 30km','30km < d <= 50km','50km < d <= 100km', '> 100km'], axis=0)

df_dist_saintCloud = pd.DataFrame(df_saintCloud.drop_duplicates('pseudo_provisoire', keep='first').groupby('distance_int').size(), columns=['count'])#.reset_index()
df_dist_saintCloud = df_dist_saintCloud.reindex(['< 1km', '1km < d <= 10km','10km < d <= 30km','30km < d <= 50km','50km < d <= 100km', '> 100km'], axis=0)


df_dist_paris
df_dist_saintCloud


print(sum(df_dist_paris["count"]))
print(sum(df_dist_saintCloud["count"]))

print(f"patients selon la distance au centre de Paris : {df_dist_paris}")
print(f"patients selon la distance au centre de SAint Cloud : {df_dist_saintCloud}")

917
2367
patients selon la distance au centre de Paris :                    count
distance_int            
< 1km                 20
1km < d <= 10km      404
10km < d <= 30km     287
30km < d <= 50km      77
50km < d <= 100km     50
> 100km               79
patients selon la distance au centre de SAint Cloud :                    count
distance_int            
< 1km                 44
1km < d <= 10km     1195
10km < d <= 30km     710
30km < d <= 50km     157
50km < d <= 100km    141
> 100km              120


## Nombre de patients de la cohorte en fonction de la distance aux centres Curie

#### Centre de Paris

In [19]:
import plotly.express as px

fig = px.bar(df_dist_paris, x=df_dist_paris.index, y="count",text='count') #, barmode="group"

fig.update_layout(title="Nombre de patients de la cohorte selon la distance a leur centre de traitement (Paris)")
fig.update_xaxes(title="Distance au centre")
fig.update_traces(textposition='outside',textfont=dict(size=16))
config = {
        'toImageButtonOptions': {
            'format': 'png',  
            'filename': 'patients_selon_dist_Paris',
            'height': 1080,
            'width': 1920,
            'scale': 6 
        }
    }

# Show the figure
fig.show(config=config)


#### Centre de Saint Cloud

In [20]:
fig = px.bar(df_dist_saintCloud, x=df_dist_saintCloud.index, y="count", text = "count") #, barmode="group"

fig.update_layout(title="Nombre de patients selon la distance a leur centre de traitement (Saint-Cloud)")
fig.update_xaxes(title="Distance au centre")
fig.update_traces(textposition = "outside",textfont=dict(size=16))
config = {
        'toImageButtonOptions': {
            'format': 'png',  # one of png, svg, jpeg, webp
            'filename': 'patients_selon_dist_StCloud',
            'height': 1080,
            'width': 1920,
            'scale': 6  # Multiply title/legend/axis/canvas sizes by this factor
        }
    }

# Show the figure
fig.show(config=config)

## Analyse du nombre d'evenements medicaux selon la distance aux differents centres de l'institut  Curie

### Centre de Paris

In [21]:
pd.DataFrame(df_em.groupby("EM").size(),columns =['count']).reset_index()

,EM,count
0,ap,1022
1,ep_mtev,694
2,ins,31
3,ira,420
4,pneumo,607
5,unap_appel,2090
6,unap_consult,1706


In [22]:
#total evenements medicaux
sum(pd.DataFrame(df_em.groupby("EM").size(),columns =['count']).reset_index()["count"])

6570

In [23]:
# Intervalles de distances
dist = ['< 1km', '1km < d <= 10km', '10km < d <= 30km', '30km < d <= 50km', '50km < d <= 100km', '> 100km']
EM = ['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult']

# Fonction pour compter les événements cliniques par distance

def compter_evenements_par_distance(df, EM, dist):
    df_em_dist = pd.DataFrame(columns=["index", "distance", "EM", "count"])
    i = 0
    for em  in EM:
        df_em = df[df['EM']== em]
        for d in dist:
            df_even_dist = df_em[df_em['distance_int'] == d]
            count_em_dist = len(df_even_dist)
            df_em_dist.at[i, "index"] = i
            df_em_dist.at[i, "distance"] = d
            df_em_dist.at[i, "EM"] = em
            df_em_dist.at[i, 'count'] = count_em_dist
            i += 1
    return df_em_dist


In [24]:
# Compter les événements pour Paris et Saint-Cloud
df_em_dist_paris = compter_evenements_par_distance(df_paris, EM, dist)
df_em_dist_saintCloud = compter_evenements_par_distance(df_saintCloud,EM,dist)

df_em_dist_paris_sans_unap = df_em_dist_paris[~((df_em_dist_paris['EM'] == 'unap_appel') | (df_em_dist_paris['EM'] == 'unap_consult'))] 
df_em_dist_saintCloud_sans_unap = df_em_dist_saintCloud[~((df_em_dist_saintCloud['EM'] == 'unap_appel') | (df_em_dist_paris['EM'] == 'unap_consult'))]

df_em_dist_saintCloud_sans_unap

,index,distance,EM,count
0,0,< 1km,ap,5
1,1,1km < d <= 10km,ap,203
2,2,10km < d <= 30km,ap,72
3,3,30km < d <= 50km,ap,15
4,4,50km < d <= 100km,ap,5
5,5,> 100km,ap,14
6,6,< 1km,ep_mtev,4
7,7,1km < d <= 10km,ep_mtev,153
8,8,10km < d <= 30km,ep_mtev,71
9,9,30km < d <= 50km,ep_mtev,20


In [25]:
df_em_dist_paris = df_em_dist_paris.pivot(index='distance', columns='EM', values='count')
df_em_dist_saintCloud = df_em_dist_saintCloud.pivot(index='distance', columns='EM', values='count')

df_em_dist_paris = df_em_dist_paris.reindex(['< 1km', '1km < d <= 10km','10km < d <= 30km','30km < d <= 50km','50km < d <= 100km', '> 100km'],axis=0)
df_em_dist_saintCloud = df_em_dist_saintCloud.reindex(['< 1km', '1km < d <= 10km','10km < d <= 30km','30km < d <= 50km','50km < d <= 100km', '> 100km'],axis=0)

df_em_dist_paris['total'] = df_em_dist_paris.sum(axis=1)
# df_em_dist_paris.loc['sum'] = df_em_dist_paris.sum()

df_em_dist_saintCloud['total'] = df_em_dist_saintCloud.sum(axis=1)
# df_em_dist_saintCloud.loc['sum'] = df_em_dist_saintCloud.sum()





df_em_dist_paris_sans_unap


,index,distance,EM,count
0,0,< 1km,ap,4
1,1,1km < d <= 10km,ap,170
2,2,10km < d <= 30km,ap,115
3,3,30km < d <= 50km,ap,30
4,4,50km < d <= 100km,ap,21
5,5,> 100km,ap,32
6,6,< 1km,ep_mtev,13
7,7,1km < d <= 10km,ep_mtev,172
8,8,10km < d <= 30km,ep_mtev,118
9,9,30km < d <= 50km,ep_mtev,32


In [26]:
df_em_dist_paris_sans_unap = df_em_dist_paris_sans_unap.pivot(index='distance', columns='EM', values='count')
df_em_dist_saintCloud_sans_unap = df_em_dist_saintCloud_sans_unap.pivot(index='distance', columns='EM', values='count')


df_em_dist_paris_sans_unap = df_em_dist_paris_sans_unap.reindex(['< 1km', '1km < d <= 10km','10km < d <= 30km','30km < d <= 50km','50km < d <= 100km', '> 100km'],axis=0)
df_em_dist_saintCloud_sans_unap = df_em_dist_saintCloud_sans_unap.reindex(['< 1km', '1km < d <= 10km','10km < d <= 30km','30km < d <= 50km','50km < d <= 100km', '> 100km'],axis=0)

df_em_dist_paris_sans_unap['total'] = df_em_dist_paris_sans_unap.sum(axis=1)
df_em_dist_saintCloud_sans_unap['total'] = df_em_dist_saintCloud_sans_unap.sum(axis=1)

df_em_dist_paris_sans_unap


EM,ap,ep_mtev,ins,ira,pneumo,total
distance,,,,,,
< 1km,4,13,0,4,7,28
1km < d <= 10km,170,172,7,111,160,620
10km < d <= 30km,115,118,7,67,109,416
30km < d <= 50km,30,32,2,13,17,94
50km < d <= 100km,21,12,3,11,25,72
> 100km,32,35,3,8,33,111


In [27]:
print(f"total d'evenements medicaux des patients ayant pour centre de traitement  Paris {sum(df_em_dist_paris['total'])}")
print(f"total d'evenements medicaux des patients ayant pour centre de traitement  Saint-Cloud {sum(df_em_dist_saintCloud['total'])}")

total d'evenements medicaux des patients ayant pour centre de traitement  Paris 1501
total d'evenements medicaux des patients ayant pour centre de traitement  Saint-Cloud 4587


In [28]:
df_em_dist_saintCloud_sans_unap

EM,ap,ep_mtev,ins,ira,pneumo,total
distance,,,,,,
< 1km,5,4,0,1,6,16
1km < d <= 10km,203,153,5,107,130,598
10km < d <= 30km,72,71,2,52,52,249
30km < d <= 50km,15,20,1,14,14,64
50km < d <= 100km,5,13,0,10,10,38
> 100km,14,9,0,5,11,39


In [29]:
df_em_dist_paris.to_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_paris.csv', sep=";")
df_em_dist_saintCloud.to_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_saint_cloud.csv',sep=";")

In [30]:
df_em_dist_paris_sans_unap.to_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_paris_sans_unap.csv', sep=";")
df_em_dist_saintCloud_sans_unap.to_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_saint_cloud_sans_unap.csv',sep=";")

In [31]:
# df_tab_dist_paris = pd.read_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_paris.csv',sep=";")
# df_tab_dist_saintcloud = pd.read_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_saint_cloud.csv',sep=";")
# #df_tab_dist = df_tab_dist.drop(0)


#df_em_dist_paris.set_index('distance', inplace=True)
df_em_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult','total']] = df_em_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult','total']].astype(int)


#df_em_dist_saintCloud.set_index('distance', inplace=True)
df_em_dist_saintCloud[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult','total']] = df_em_dist_saintCloud[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult','total']].astype(int)

#df_tab_dist = df_tab_dist.drop('total',axis=1)

df_em_dist_paris_sans_unap = df_em_dist_paris_sans_unap[['ap','ep_mtev','ins', 'ira', 'pneumo','total']].astype(int)
df_em_dist_saintCloud_sans_unap = df_em_dist_saintCloud_sans_unap[['ap','ep_mtev','ins', 'ira', 'pneumo','total']].astype(int)




In [32]:
df_em_dist_paris_sans_unap

EM,ap,ep_mtev,ins,ira,pneumo,total
distance,,,,,,
< 1km,4,13,0,4,7,28
1km < d <= 10km,170,172,7,111,160,620
10km < d <= 30km,115,118,7,67,109,416
30km < d <= 50km,30,32,2,13,17,94
50km < d <= 100km,21,12,3,11,25,72
> 100km,32,35,3,8,33,111


In [33]:
df_em_dist_saintCloud_sans_unap

EM,ap,ep_mtev,ins,ira,pneumo,total
distance,,,,,,
< 1km,5,4,0,1,6,16
1km < d <= 10km,203,153,5,107,130,598
10km < d <= 30km,72,71,2,52,52,249
30km < d <= 50km,15,20,1,14,14,64
50km < d <= 100km,5,13,0,10,10,38
> 100km,14,9,0,5,11,39


#### Heatmap en nombres absolus centre Paris

In [34]:
import seaborn as sns
%matplotlib inline


# sns.heatmap(df_tab_dist, linewidths=0.5, annot=False)

# df_tab_dist.style.background_gradient(cmap='Blues')

fig = px.imshow(df_em_dist_paris.drop(columns='total'), text_auto=True, color_continuous_scale='magma_r')
fig.show()



#### Heatmap en nombres absolus centre de Saint Cloud

In [35]:
import seaborn as sns
%matplotlib inline


# sns.heatmap(df_tab_dist, linewidths=0.5, annot=False)

# df_tab_dist.style.background_gradient(cmap='Blues')

fig = px.imshow(df_em_dist_saintCloud.drop(columns='total'), text_auto=True, color_continuous_scale='magma_r')
fig.show()




##### Heatmap en pourcentage centre de paris

In [36]:
# Lecture du fichier CSV
df_tab_dist_paris = pd.read_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_paris.csv', sep=";")

# Configuration de l'index et conversion des colonnes en entiers
df_tab_dist_paris.index = df_tab_dist_paris['distance']
df_tab_dist_paris = df_tab_dist_paris.drop('distance', axis=1)
df_tab_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult', 'total']] = df_tab_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult', 'total']].astype(int)

# Calcul de la somme des patients pour chaque ligne (sans 'total')
df_tab_dist_paris['Sum'] = df_tab_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult']].sum(axis=1)

# Diviser chaque colonne par la colonne 'total'
for column in ['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult']:
    df_tab_dist_paris[column] = ((df_tab_dist_paris[column] / df_tab_dist_paris['total'])*100).round(2)

# Création de la heatmap sans la colonne 'total' et la colonne fictive pour espacement
fig = px.imshow(df_tab_dist_paris.drop(['total', 'Sum'], axis=1), text_auto=True, color_continuous_scale='magma_r')


# Ajout de la somme des patients à la heatmap
for i, sum_value in enumerate(df_tab_dist_paris['Sum']):
    pourcentage = (sum_value/df_tab_dist_paris['total'].sum())*100
    fig.add_annotation(
        dict(
            text=f"N ={sum_value}/{df_tab_dist_paris['total'].sum()}: {pourcentage:.2f}%",
            x=len(df_tab_dist_paris.columns) - 2.2,  # Placer un peu à droite de la dernière colonne de données
            y= i - 0.1, 
            xref="x", 
            yref="y", 
            showarrow=False, 
            font=dict(color="black", size=12),
            xanchor="left",
            yanchor = "middle"
        )
    )

fig.add_annotation(
    dict(
        text="n = nombre de patients",
        x=1.115,  # Position horizontale au milieu de la heatmap
        y=1,  # Position verticale juste au-dessus de la heatmap
        xref="paper",  # Référence de la position horizontale
        yref="paper",  # Référence de la position verticale
        showarrow=False,  # Pas de flèche
        font=dict(color="black", size=12),
        xanchor="center",  # Alignement horizontal au centre
        yanchor="bottom"  # Alignement vertical en bas
    )
)

# Ajustement des marges pour la légende
fig.update_layout(
    margin=dict(l=30, r=250, t=30, b=30),  # Augmenter 'r' pour laisser plus de place pour la légende
    xaxis_title="Pathologies",
    yaxis_title="Distance",
    legend_title_text='Pathologies'
)

fig.show()


#### Heatmap en pourcentage centre de Saint Cloud

In [37]:
df_tab_dist_saintcloud = pd.read_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_saint_cloud.csv', sep=";")

# Configuration de l'index et conversion des colonnes en entiers
df_tab_dist_saintcloud.index = df_tab_dist_saintcloud['distance']
df_tab_dist_saintcloud = df_tab_dist_saintcloud.drop('distance', axis=1)
df_tab_dist_saintcloud[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult', 'total']] = df_tab_dist_saintcloud[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult','total']].astype(int)

# Calcul de la somme des patients pour chaque ligne (sans 'total')
df_tab_dist_saintcloud['Sum'] = df_tab_dist_saintcloud[['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult']].sum(axis=1)

# Diviser chaque colonne par la colonne 'total'
for column in ['ap','ep_mtev','ins', 'ira', 'pneumo', 'unap_appel','unap_consult']:
    df_tab_dist_saintcloud[column] = ((df_tab_dist_saintcloud[column] / df_tab_dist_saintcloud['total'])*100).round(2)

# Création de la heatmap sans la colonne 'total' et la colonne fictive pour espacement
fig = px.imshow(df_tab_dist_saintcloud.drop(['total', 'Sum'], axis=1), text_auto=True, color_continuous_scale='magma_r')


# Ajout de la somme des patients à la heatmap
for i, sum_value in enumerate(df_tab_dist_saintcloud['Sum']):
    pourcentage = (sum_value/df_tab_dist_paris['total'].sum())*100
    fig.add_annotation(
        dict(
            text=f"N ={sum_value}/{df_tab_dist_saintcloud['total'].sum()}: {pourcentage:.2f}%",
            x=len(df_tab_dist_saintcloud.columns) - 2.2,  # Placer un peu à droite de la dernière colonne de données
            y= i - 0.1, 
            xref="x", 
            yref="y", 
            showarrow=False, 
            font=dict(color="black", size=12),
            xanchor="left",
            yanchor = "middle"
        )
    )

fig.add_annotation(
    dict(
        text="n = nombre de patients",
        x=1.115,  # Position horizontale au milieu de la heatmap
        y=1,  # Position verticale juste au-dessus de la heatmap
        xref="paper",  # Référence de la position horizontale
        yref="paper",  # Référence de la position verticale
        showarrow=False,  # Pas de flèche
        font=dict(color="black", size=12),
        xanchor="center",  # Alignement horizontal au centre
        yanchor="bottom"  # Alignement vertical en bas
    )
)

# Ajustement des marges pour la légende
fig.update_layout(
    margin=dict(l=30, r=250, t=30, b=30),  # Augmenter 'r' pour laisser plus de place pour la légende
    xaxis_title="Pathologies",
    yaxis_title="Distance",
    legend_title_text='Pathologies'
)

fig.show()


### Heatmap saint cloud sans appel et consultation

In [38]:
#saint cloud

import seaborn as sns
%matplotlib inline


# sns.heatmap(df_tab_dist, linewidths=0.5, annot=False)

# df_tab_dist.style.background_gradient(cmap='Blues')

fig = px.imshow(df_em_dist_saintCloud_sans_unap.drop(columns='total'), text_auto=True, color_continuous_scale='magma_r')
fig.show()

In [39]:
df_tab_dist_saintcloud = pd.read_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_saint_cloud_sans_unap.csv', sep=";")

# Configuration de l'index et conversion des colonnes en entiers
df_tab_dist_saintcloud.index = df_tab_dist_saintcloud['distance']
df_tab_dist_saintcloud = df_tab_dist_saintcloud.drop('distance', axis=1)
df_tab_dist_saintcloud[['ap','ep_mtev','ins', 'ira', 'pneumo', 'total']] = df_tab_dist_saintcloud[['ap','ep_mtev','ins', 'ira', 'pneumo','total']].astype(int)

# Calcul de la somme des patients pour chaque ligne (sans 'total')
df_tab_dist_saintcloud['Sum'] = df_tab_dist_saintcloud[['ap','ep_mtev','ins', 'ira', 'pneumo']].sum(axis=1)

# Diviser chaque colonne par la colonne 'total'
for column in ['ap','ep_mtev','ins', 'ira', 'pneumo']:
    df_tab_dist_saintcloud[column] = ((df_tab_dist_saintcloud[column] / df_tab_dist_saintcloud['total'])*100).round(2)

# Création de la heatmap sans la colonne 'total' et la colonne fictive pour espacement
fig = px.imshow(df_tab_dist_saintcloud.drop(['total', 'Sum'], axis=1), text_auto=True, color_continuous_scale='magma_r')


# Ajout de la somme des patients à la heatmap
for i, sum_value in enumerate(df_tab_dist_saintcloud['Sum']):
    pourcentage = (sum_value/df_tab_dist_saintcloud['total'].sum())*100
    fig.add_annotation(
        dict(
            text=f"N ={sum_value}/{df_tab_dist_saintcloud['total'].sum()}: {pourcentage:.2f}%",
            x=len(df_tab_dist_saintcloud.columns) - 2.2,  # Placer un peu à droite de la dernière colonne de données
            y= i - 0.1, 
            xref="x", 
            yref="y", 
            showarrow=False, 
            font=dict(color="black", size=12),
            xanchor="left",
            yanchor = "middle"
        )
    )

fig.add_annotation(
    dict(
        text="n = nombre de patients",
        x=1.115,  # Position horizontale au milieu de la heatmap
        y=1,  # Position verticale juste au-dessus de la heatmap
        xref="paper",  # Référence de la position horizontale
        yref="paper",  # Référence de la position verticale
        showarrow=False,  # Pas de flèche
        font=dict(color="black", size=12),
        xanchor="center",  # Alignement horizontal au centre
        yanchor="bottom"  # Alignement vertical en bas
    )
)

# Ajustement des marges pour la légende
fig.update_layout(
    margin=dict(l=30, r=250, t=30, b=30),  # Augmenter 'r' pour laisser plus de place pour la légende
    xaxis_title="Pathologies",
    yaxis_title="Distance",
    legend_title_text='Pathologies'
)

fig.show()


In [40]:
#saint cloud

import seaborn as sns
%matplotlib inline


# sns.heatmap(df_tab_dist, linewidths=0.5, annot=False)

# df_tab_dist.style.background_gradient(cmap='Blues')

fig = px.imshow(df_em_dist_paris_sans_unap.drop(columns='total'), text_auto=True, color_continuous_scale='magma_r')
fig.show()

In [41]:
df_tab_dist_paris = pd.read_csv('../geocodeur/from_hegp/data/data_octobre_2023/dist_em_paris_sans_unap.csv', sep=";")

# Configuration de l'index et conversion des colonnes en entiers
df_tab_dist_paris.index = df_tab_dist_paris['distance']
df_tab_dist_paris = df_tab_dist_paris.drop('distance', axis=1)
df_tab_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo', 'total']] = df_tab_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo','total']].astype(int)

# Calcul de la somme des patients pour chaque ligne (sans 'total')
df_tab_dist_paris['Sum'] = df_tab_dist_paris[['ap','ep_mtev','ins', 'ira', 'pneumo']].sum(axis=1)

# Diviser chaque colonne par la colonne 'total'
for column in ['ap','ep_mtev','ins', 'ira', 'pneumo']:
    df_tab_dist_paris[column] = ((df_tab_dist_paris[column] / df_tab_dist_paris['total'])*100).round(2)

# Création de la heatmap sans la colonne 'total' et la colonne fictive pour espacement
fig = px.imshow(df_tab_dist_paris.drop(['total', 'Sum'], axis=1), text_auto=True, color_continuous_scale='magma_r')


# Ajout de la somme des patients à la heatmap
for i, sum_value in enumerate(df_tab_dist_paris['Sum']):
    pourcentage = (sum_value/df_tab_dist_paris['total'].sum())*100
    fig.add_annotation(
        dict(
            text=f"N ={sum_value}/{df_tab_dist_paris['total'].sum()}: {pourcentage:.2f}%",
            x=len(df_tab_dist_paris.columns) - 2.2,  # Placer un peu à droite de la dernière colonne de données
            y= i - 0.1, 
            xref="x", 
            yref="y", 
            showarrow=False, 
            font=dict(color="black", size=12),
            xanchor="left",
            yanchor = "middle"
        )
    )

fig.add_annotation(
    dict(
        text="n = nombre de patients",
        x=1.115,  # Position horizontale au milieu de la heatmap
        y=1,  # Position verticale juste au-dessus de la heatmap
        xref="paper",  # Référence de la position horizontale
        yref="paper",  # Référence de la position verticale
        showarrow=False,  # Pas de flèche
        font=dict(color="black", size=12),
        xanchor="center",  # Alignement horizontal au centre
        yanchor="bottom"  # Alignement vertical en bas
    )
)

# Ajustement des marges pour la légende
fig.update_layout(
    margin=dict(l=30, r=250, t=30, b=30),  # Augmenter 'r' pour laisser plus de place pour la légende
    xaxis_title="Pathologies",
    yaxis_title="Distance",
    legend_title_text='Pathologies'
)

fig.show()

REPARTITION DES EVENEMENTS SELON LA DISTANCE

In [42]:
df_combined = df_em_dist_paris_sans_unap.add(df_em_dist_saintCloud_sans_unap, fill_value=0)
df_combined

EM,ap,ep_mtev,ins,ira,pneumo,total
distance,,,,,,
< 1km,9,17,0,5,13,44
1km < d <= 10km,373,325,12,218,290,1218
10km < d <= 30km,187,189,9,119,161,665
30km < d <= 50km,45,52,3,27,31,158
50km < d <= 100km,26,25,3,21,35,110
> 100km,46,44,3,13,44,150


In [43]:
df_combined.to_csv('../geocodeur/from_hegp/data/data_octobre_2023/df_em_dist_sans_unap.csv',sep=";")

In [44]:
df_combined = pd.read_csv('../geocodeur/from_hegp/data/data_octobre_2023/df_em_dist_sans_unap.csv', sep=";")
# Configuration de l'index et conversion des colonnes en entiers
df_combined.index = df_combined['distance']
df_combined = df_combined.drop('distance', axis=1)
df_combined[['ap','ep_mtev','ins', 'ira', 'pneumo', 'total']] = df_combined[['ap','ep_mtev','ins', 'ira', 'pneumo','total']].astype(int)

# Calcul de la somme des patients pour chaque ligne (sans 'total')
df_combined['Sum'] = df_combined[['ap','ep_mtev','ins', 'ira', 'pneumo']].sum(axis=1)

# Diviser chaque colonne par la colonne 'total'
for column in ['ap','ep_mtev','ins', 'ira', 'pneumo']:
    df_combined[column] = ((df_combined[column] / df_combined['total'])*100).round(2)

# Création de la heatmap sans la colonne 'total' et la colonne fictive pour espacement
fig = px.imshow(df_combined.drop(['total', 'Sum'], axis=1), text_auto=True, color_continuous_scale='magma_r')


# Ajout de la somme des patients à la heatmap
for i, sum_value in enumerate(df_combined['Sum']):
    pourcentage = (sum_value/df_combined['total'].sum())*100
    fig.add_annotation(
        dict(
            text=f"N ={sum_value}/{df_combined['total'].sum()}: {pourcentage:.2f}%",
            x=len(df_combined.columns) - 2.2,  
            y= i - 0.1, 
            xref="x", 
            yref="y", 
            showarrow=False, 
            font=dict(color="black", size=12),
            xanchor="left",
            yanchor = "middle"
        )
    )

fig.add_annotation(
    dict(
        text="n = nombre de patients",
        x=1.115,  # Position horizontale au milieu de la heatmap
        y=1,  # Position verticale juste au-dessus de la heatmap
        xref="paper",  # Référence de la position horizontale
        yref="paper",  # Référence de la position verticale
        showarrow=False,  # Pas de flèche
        font=dict(color="black", size=12),
        xanchor="center",  # Alignement horizontal au centre
        yanchor="bottom"  # Alignement vertical en bas
    )
)

# Ajustement des marges pour la légende
fig.update_layout(
    margin=dict(l=30, r=250, t=30, b=30),  
    xaxis_title="Pathologies",
    yaxis_title="Distance",
    legend_title_text='Pathologies'
)

fig.show()
